In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [22]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 3.97


### Load Player Data and Bookmaker Data

In [23]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_13589/1321450873.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,John Collins,Over,14.0,-137,2025-11-21,2025-11-20T23:41:20Z
1,PrizePicks,player_points,John Collins,Under,14.0,-137,2025-11-21,2025-11-20T23:41:20Z
2,PrizePicks,player_points,James Harden,Over,27.5,-137,2025-11-21,2025-11-20T23:41:20Z
3,PrizePicks,player_points,James Harden,Under,27.5,-137,2025-11-21,2025-11-20T23:41:20Z
4,PrizePicks,player_points,Franz Wagner,Over,23.5,-137,2025-11-21,2025-11-20T23:41:20Z


### Update projected starting lineups

In [7]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 8 teams with confirmed lineups


### Top EVs for single bets

In [11]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
edge_threshold=0.30, stake=10, variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, 
max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...
Pre-computing predictions for 58 unique players...
Error getting prediction for Paul George: float division by zero


/Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/pipeline.py:364: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/pipeline.py:364: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/pipeline.py:364: SettingWithCopyWarning: 
A value is trying to be set on a cop

,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,KELLY_FRACTION,SIGMA FLAG
0,Bobby Portis,FanDuel,14.5,9.42,Under,-104,1,5.41,0.562,High
1,Bobby Portis,BetRivers,13.5,9.42,Under,105,0,4.82,0.459,High
2,Bobby Portis,DraftKings,14.5,9.42,Under,-112,1,4.73,0.530,High
3,Bobby Portis,BetMGM,13.5,9.42,Under,100,0,4.47,0.447,High
4,Bobby Portis,BetRivers,14.5,9.42,Under,-120,1,4.36,0.523,High


## Top EVs for 2 leg bets

### Underdog picks

In [25]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['COMMENCE_TIME'] == '2025-11-21')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 38 players...
Error getting prediction for Paul George: float division by zero
Processing 32 players with valid predictions...
Generated 440 valid 2-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 31 combinations from 440 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Tyus Jones,Bobby Portis,3.5,14.5,4.46,9.42,over,under,0,4.77,0.239,Low,High
1,Tristan da Silva,Bobby Portis,11.5,14.5,14.73,9.42,over,under,0,4.67,0.233,High,High
2,Tyus Jones,Kobe Sanders,3.5,9.5,4.46,12.70,over,over,0,3.48,0.174,Low,High
3,Kobe Sanders,Santi Aldama,9.5,17.5,12.70,13.80,over,under,0,3.30,0.165,High,High
4,Tristan da Silva,Santi Aldama,11.5,17.5,14.73,13.80,over,under,0,3.02,0.151,High,High
5,Luke Kennard,VJ Edgecombe,5.5,14.5,7.28,17.21,over,over,0,2.25,0.112,Med,High
6,Nickeil Alexander-Walker,VJ Edgecombe,16.5,14.5,19.65,17.21,over,over,0,1.57,0.079,High,High
7,Luke Kennard,Malik Monk,5.5,13.5,7.28,11.09,over,under,0,1.54,0.077,Med,High
8,Nickeil Alexander-Walker,Malik Monk,16.5,13.5,19.65,11.09,over,under,0,1.15,0.058,High,High
9,Keldon Johnson,Zach Edey,14.5,13.5,12.54,14.88,under,over,0,-0.17,0.000,High,Med


### Prizepicks picks

In [26]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')& (dfsData['COMMENCE_TIME'] == '2025-11-21')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

Pre-computing predictions for 58 players...
Error getting prediction for Paul George: float division by zero
Processing 52 players with valid predictions...
Generated 1180 valid 2-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 51 combinations from 1180 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Santi Aldama,Bobby Portis,18.5,14.5,13.80,9.42,under,under,1,5.66,0.283,High,High
1,Kobe Sanders,Bobby Portis,9.5,14.5,12.70,9.42,over,under,0,4.95,0.247,High,High
2,Tyus Jones,Santi Aldama,3.5,18.5,4.46,13.80,over,under,0,4.06,0.203,Low,High
3,Kobe Sanders,Tyus Jones,9.5,3.5,12.70,4.46,over,over,0,3.47,0.174,High,Low
4,Brook Lopez,Goga Bitadze,6.0,4.5,7.87,5.94,over,over,0,2.97,0.149,Med,Low


## 3 leg parlay

### Underdog picks

In [27]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['COMMENCE_TIME'] == '2025-11-21')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 38 players...
Error getting prediction for Paul George: float division by zero
Processing 32 players with valid predictions...
Generated 4666 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 20 combinations from 4666 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Tyus Jones,Kobe Sanders,Bobby Portis,3.5,9.5,14.5,4.46,12.70,9.42,over,over,under,0,8.76,0.175,Low,High,High
1,Tyus Jones,Santi Aldama,Bobby Portis,3.5,17.5,14.5,4.46,13.80,9.42,over,under,under,0,8.55,0.171,Low,High,High
2,Kobe Sanders,Luke Kennard,Santi Aldama,9.5,5.5,17.5,12.70,7.28,13.80,over,over,under,0,6.36,0.127,High,Med,High
3,Tristan da Silva,Luke Kennard,Nickeil Alexander-Walker,11.5,5.5,16.5,14.73,7.28,19.65,over,over,over,0,5.41,0.108,High,Med,High
4,Tristan da Silva,Nickeil Alexander-Walker,VJ Edgecombe,11.5,16.5,14.5,14.73,19.65,17.21,over,over,over,0,4.52,0.090,High,High,High


### Prizepicks picks

In [28]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')& (dfsData['COMMENCE_TIME'] == '2025-11-21')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 58 players...
Error getting prediction for Paul George: float division by zero
Processing 52 players with valid predictions...
Generated 20896 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 34 combinations from 20896 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Tyus Jones,Santi Aldama,Bobby Portis,3.5,18.5,14.5,4.46,13.80,9.42,over,under,under,0,9.85,0.197,Low,High,High
1,Kobe Sanders,Santi Aldama,Bobby Portis,9.5,18.5,14.5,12.70,13.80,9.42,over,under,under,0,9.82,0.196,High,High,High
2,Kobe Sanders,Tyus Jones,Luke Kennard,9.5,3.5,5.5,12.70,4.46,7.28,over,over,over,0,6.57,0.131,High,Low,Med
3,Tristan da Silva,Goga Bitadze,Luke Kennard,11.5,4.5,5.5,14.73,5.94,7.28,over,over,over,0,6.05,0.121,High,Low,Med
4,Tristan da Silva,Goga Bitadze,Nickeil Alexander-Walker,11.5,4.5,16.5,14.73,5.94,19.65,over,over,over,0,5.48,0.110,High,Low,High
